# CIFAR-100 experiments

**Everything is pre-configured. You do not need GitHub, a zip file, or any dataset.**

1. Accelerator: **GPU T4 x2**
2. Internet: **On**
3. Inputs: **leave empty**
4. **Save Version -> Save & Run All (Commit)**

The notebook clones the code itself, downloads CIFAR-100 itself, and trains the
distillation teacher inside the session.

Run the sessions one after another. Between them, change `SESSION` in the next
cell. That single digit is the only thing you ever edit.

| SESSION | runs | hours | what | status |
|---|---|---|---|---|
| 1 | D1, D3, D5 | ~6.0 | teacher at seed 42, then both distilled models | done |
| 2 | D2, D4 | ~3.8 | the same two models, cross-entropy | done |
| 3 | D6, D7, D8 | ~6.1 | the clustering ablation at seed 42 | done |
| 4 | D19, D20, D21 | ~6.3 | the same ablation at seed 43 | done |
| 5 | D9, D10, D11 | ~6.6 | teacher at seed 43, then both distilled models | done |
| 6 | D12, D13 | ~4.5 | the same two models, cross-entropy, seed 43 | **run this next** |
| 7 | D14, D15, D16 | ~6.6 | teacher at seed 44, then both distilled models | |
| 8 | D17, D18 | ~4.5 | the same two models, cross-entropy, seed 44 | |

Sessions 1 to 5 are finished and nothing needs re-running.

Sessions 5 to 8 bring the main CIFAR-100 comparison --- our model against
ShuffleNetV2-x1.5, with and without distillation --- to three seeds. That
comparison is the one the paper reports, and it is currently at one seed.

Sessions 5 and 7 must keep the teacher first: it is the checkpoint the two
distilled runs load, and it does not survive from one session to the next.
Sessions 6 and 8 are cross-entropy only, so their order does not matter.

When a session finishes, download `D_sN_results.zip` from the output tab.


In [ ]:
# ---- The only line you edit: 6, then 7, then 8 -----------------------------------------
SESSION = 6

# ---- Everything below is derived; leave it alone ----------------------------------------
_SESSIONS = {
    1: ("D_s1", ["D1", "D3", "D5"]),        # done: teacher s42, then the two KD runs
    2: ("D_s2", ["D2", "D4"]),              # done: the same two models, cross-entropy
    3: ("D_s3", ["D6", "D7", "D8"]),        # done: rho sweep at seed 42
    4: ("D_s4", ["D19", "D20", "D21"]),     # done: rho sweep at seed 43
    5: ("D_s5", ["D9", "D10", "D11"]),      # done: teacher s43, then both KD runs
    6: ("D_s6", ["D12", "D13"]),            # both CE runs at seed 43, no teacher needed
    7: ("D_s7", ["D14", "D15", "D16"]),     # teacher s44 first, then both KD runs
    8: ("D_s8", ["D17", "D18"]),            # both CE runs at seed 44, no teacher needed
}
assert SESSION in _SESSIONS, f"SESSION must be 1-8; got {SESSION!r}"
SESSION_NAME, QUEUE_GPU0 = _SESSIONS[SESSION]
QUEUE_GPU1 = []
DEADLINE_HOURS = 11.0
REPO_URL = "https://github.com/nvhungus/Lightweight_Context-Cluster.git"
REPO_BRANCH = "resubmission"
print(f"session {SESSION}: {SESSION_NAME}, queue {QUEUE_GPU0}")

In [ ]:
import json, os, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
ROOT = WORK / "Lightweight-Context-Cluster"
RUNS = WORK / "runs"
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
print("commit:", subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

import torch
sys.path.insert(0, str(ROOT / "tools"))
import run_queue

N_GPUS = torch.cuda.device_count()
assert N_GPUS >= 1 and "T4" in torch.cuda.get_device_name(0), "Select the GPU T4 x2 accelerator."
if QUEUE_GPU1 and N_GPUS < 2:
    raise SystemExit("QUEUE_GPU1 is set but only one GPU is visible: select GPU T4 x2.")

plan = run_queue.load_plan()
ids = QUEUE_GPU0 + QUEUE_GPU1
unknown = [i for i in ids if i not in plan["by_id"]]
assert not unknown, f"unknown run IDs: {unknown}"
assert len(ids) == len(set(ids)), "a run ID appears twice"
for name, queue in [("GPU0", QUEUE_GPU0), ("GPU1", QUEUE_GPU1)]:
    if queue:
        # Same rule as run_queue.py --deadline-hours: start a run only if elapsed + 1.3 x estimate fits.
        elapsed, deferred = 0.0, []
        for i in queue:
            est = run_queue.estimated_hours(plan, plan["by_id"][i]) or 0.0
            if elapsed + run_queue.DEADLINE_MARGIN * est > DEADLINE_HOURS:
                deferred.append(i)
            else:
                elapsed += est
        print(f"{name}: {queue}  estimated {elapsed:.1f} h"
              + (f"  -> will be deferred if estimates hold: {deferred}" if deferred else "  -> all fit before the deadline"))


def find_tin_root() -> Path | None:
    for depth in range(1, 7):
        for hit in Path("/kaggle/input").glob("/".join(["*"] * depth) + "/wnids.txt"):
            if (hit.parent / "train").is_dir() and (hit.parent / "val" / "val_annotations.txt").exists():
                return hit.parent
    return None


needs_tin = any(plan["by_id"][i].dataset == "tin" for i in ids)
TIN_ROOT = find_tin_root()
assert TIN_ROOT or not needs_tin, "Attach the Tiny-ImageNet dataset (akash2sharma/tiny-imagenet)."
CIFAR_ROOT = WORK / "data"
print("TIN_ROOT =", TIN_ROOT)

In [ ]:
def queue_cmd(queue, gpu, workers):
    return [sys.executable, "tools/run_queue.py", "--ids", *queue, "--output", str(RUNS),
            "--tin-root", str(TIN_ROOT), "--cifar-root", str(CIFAR_ROOT), "--search-dir", "/kaggle/input",
            "--gpu", str(gpu), "--workers", str(workers), "--deadline-hours", str(DEADLINE_HOURS), "--keep-going"]


# One queue: 4 data workers (as in the first-submission timings). Two queues share 4 CPUs: 2 each.
workers = 2 if QUEUE_GPU1 else 4
jobs = []
for gpu, queue in [(0, QUEUE_GPU0), (1, QUEUE_GPU1)]:
    if queue:
        log = open(WORK / f"{SESSION_NAME}_gpu{gpu}.log", "w")
        jobs.append((gpu, subprocess.Popen(queue_cmd(queue, gpu, workers), cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, text=True), log))
        print(f"started GPU{gpu}: {queue}")

start = time.time()
while any(p.poll() is None for _, p, _ in jobs):
    time.sleep(600)
    for gpu, p, _ in jobs:
        lines = (WORK / f"{SESSION_NAME}_gpu{gpu}.log").read_text(errors="replace").splitlines()
        progress = [l for l in lines if l.startswith(("===", "epoch=", "test ", "[defer]", "[skip]", "[fail]"))]
        print(f"[{(time.time() - start) / 3600:5.2f} h] GPU{gpu}: {progress[-1] if progress else '(starting)'}", flush=True)
for gpu, p, log in jobs:
    log.close()
    print(f"\n===== GPU{gpu} queue finished with exit code {p.returncode} =====")
    text = (WORK / f"{SESSION_NAME}_gpu{gpu}.log").read_text(errors="replace")
    print(text[text.rfind("=== queue summary ==="):] if "=== queue summary ===" in text else text[-3000:])

In [ ]:
# ---- Session summary -------------------------------------------------------------------------
rows = []
for run_id in ids:
    run = plan["by_id"][run_id]
    d = RUNS / run.experiment
    test, info = d / "test_metrics.json", d / "run_info.json"
    if test.exists():
        t = json.loads(test.read_text())
        i = json.loads(info.read_text())
        rows.append((run_id, run.experiment, "done", i.get("best_val_acc1"), t.get("test_acc1"), i.get("best_epoch"), i.get("git_commit", "")[:7]))
    else:
        rows.append((run_id, run.experiment, "NOT DONE", None, None, None, ""))
print(f"{'id':>4} {'experiment':38s} {'status':9s} {'best val':>9s} {'test':>7s} {'best ep':>8s} commit")
for r in rows:
    fmt = lambda v: f"{v:.2f}" if isinstance(v, float) else "-"
    print(f"{r[0]:>4} {r[1]:38s} {r[2]:9s} {fmt(r[3]):>9s} {fmt(r[4]):>7s} {str(r[5]) if r[5] is not None else '-':>8s} {r[6]}")
not_done = [r[0] for r in rows if r[2] != "done"]
print("\nAll runs finished." if not not_done else f"\nNot finished (re-queue in the next session): {not_done}")

bundle = WORK / f"{SESSION_NAME}_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in RUNS.rglob("*"):
        if f.is_file() and f.suffix != ".pth":
            zf.write(f, f.relative_to(WORK))
    for f in WORK.glob(f"{SESSION_NAME}_gpu*.log"):
        zf.write(f, f.relative_to(WORK))
print("download:", bundle)